In [ ]:
# imports

import pandas as pd
import plotly.express as px
import plotly.io as pio

# open figures in a full browser tab instead of the cramped notebook output
# pane; switch back to "vscode" to render inline again
pio.renderers.default = "browser"

In [ ]:
from pathlib import Path

# Folder holding one exported CSV per listener
DATA_DIR = Path("data")

# Encodings to try in order; utf-8-sig also strips the BOM some exports have
ENCODINGS = ["utf-8-sig", "utf-16", "cp1252", "latin-1"]

# The personal exports and the public expansion dataset describe the same
# things under different names, so map the expansion columns onto the export
# schema as the file is read
EXPANSION_COLUMNS = {
    "track_id": "Track URI",
    "track_name": "Track Name",
    "album_name": "Album Name",
    "artists": "Artist Name(s)",
    "popularity": "Popularity",
    "duration_ms": "Duration (ms)",
    "explicit": "Explicit",
    "track_genre": "Genres",
    "danceability": "Danceability",
    "energy": "Energy",
    "key": "Key",
    "loudness": "Loudness",
    "mode": "Mode",
    "speechiness": "Speechiness",
    "acousticness": "Acousticness",
    "instrumentalness": "Instrumentalness",
    "liveness": "Liveness",
    "valence": "Valence",
    "tempo": "Tempo",
    "time_signature": "Time Signature",
}

# Present in the exports, absent from the expansion dataset
EXPANSION_MISSING = ["Release Date", "Added By", "Added At", "Record Label"]


def read_csv_any_encoding(path):
    """Read a CSV, trying each encoding until one works."""
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding)
        except (UnicodeError, pd.errors.ParserError):
            # Wrong encoding for this file, try the next one
            continue
    raise ValueError(f"Could not read {path} with any of {ENCODINGS}")


def normalize_expansion(df):
    """Reshape the public expansion CSV to match the personal exports."""
    # keep only the columns that map onto the export schema, renamed
    df = df.rename(columns=EXPANSION_COLUMNS)[list(EXPANSION_COLUMNS.values())]

    # bare ids there, full URIs in the exports
    df["Track URI"] = "spotify:track:" + df["Track URI"].astype(str)

    # several artists are joined with ";" there and "," in the exports
    df["Artist Name(s)"] = (
        df["Artist Name(s)"].astype(
            "string").str.replace(";", ",", regex=False)
    )

    # that dataset carries one row per track-genre pair, so a song in several
    # genres repeats; collapse to one row per track with the genres comma
    # joined the way the exports list them
    genres = (
        df.groupby("Track URI")["Genres"]
        .apply(lambda values: ",".join(sorted(set(values.dropna()))))
        .rename("Genres")
    )
    df = (
        df.drop(columns="Genres")
        .drop_duplicates(subset="Track URI")
        .merge(genres, on="Track URI")
    )

    # the export only columns have no counterpart here
    for column in EXPANSION_MISSING:
        df[column] = pd.NA
    return df


# Load every CSV in the data folder and tag its rows with the listener
frames = []
for path in sorted(DATA_DIR.glob("*.csv")):
    df = read_csv_any_encoding(path)
    if "track_id" in df.columns:
        # the public expansion dataset, not one of our own exports
        df = normalize_expansion(df)
        listener = "Expansion"
    else:
        # Listener name is the first word of the file name,
        # e.g. "Logan Exported Data.csv" -> "Logan"
        listener = path.stem.split()[0]
    df["Listener"] = listener
    print(f"{path.name}: {len(df)} rows as {listener}")
    frames.append(df)

# Stack all listeners into one DataFrame
data = pd.concat(frames, ignore_index=True)

# Quick sanity check: total size and rows per listener
print(data.shape)
print(data["Listener"].value_counts())
data.head()

### Data Cleaning

In [ ]:
# check for duplicate Track URIs before anything downstream relies on them

# A listener's export can list the same song more than once, e.g. when it
# sits in several of their playlists, so count the repeats per listener
# dropna=False so rows with no URI at all show up instead of vanishing
print(f"{data['Track URI'].isna().sum()} rows have no Track URI")
uri_counts = data.groupby(["Listener", "Track URI"], dropna=False).size()
repeats = uri_counts[uri_counts > 1]
print(f"{len(repeats)} Track URIs appear more than once for the same listener")
print(f"{repeats.sum() - len(repeats)} extra rows to drop")

# show a few of the worst offenders so we can eyeball what they are
if not repeats.empty:
    worst = repeats.sort_values(ascending=False).head(10).index
    display(
        data[
            [
                pair in set(worst)
                for pair in zip(data["Listener"], data["Track URI"])
            ]
        ].sort_values(["Listener", "Track URI"])[
            ["Listener", "Track URI", "Track Name", "Artist Name(s)"]
        ]
    )

# keep the first copy of each song per listener; the song level columns are
# identical across copies, and the Listener flags in a later cell still need
# one row per track-listener pair
data = data.drop_duplicates(subset=["Listener", "Track URI"])
print(data.shape)

In [ ]:
# identify missing values in each column
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# handle the 1 missing values row first

# display the row with missing values - NOT THE GENRES MISSING VALUES
data[data.isnull().any(axis=1) & data['Mode'].isnull()]

In [ ]:
# the episode row above is missing its whole audio-feature block, so drop by null
# count instead of by URI to catch any other rows that are just as incomplete

# Added By and Genres are missing for plenty of good rows, so ignore them here
null_counts = data.drop(columns=['Added By', 'Genres']).isnull().sum(axis=1)
null_drop_threshold = 10
bad_rows = null_counts >= null_drop_threshold

print(f"dropping {bad_rows.sum()} row(s) with {null_drop_threshold}+ nulls")
print(data.loc[bad_rows, ['Track URI', 'Track Name', 'Listener']])
data = data[~bad_rows]

In [ ]:
# rerun missing values check

missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# drop added by column as this is an unneeded column for our analysis
data = data.drop(columns=['Added By'])

In [ ]:
# rerun missing values check
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# inspect missing values in the genres column
data[data['Genres'].isnull()]

# add a flag for missing genres
data['Missing Genres'] = data['Genres'].isnull()


def to_list(value):
    """Split a comma-separated cell into a clean list; missing becomes []."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",") if part.strip()]


# Artist Name(s) and Genres each hold several values per track, so store them
# as lists, and keep a text version for hover labels and display
for column in ["Artist Name(s)", "Genres"]:
    data[column] = data[column].apply(to_list)
    data[column + " Text"] = data[column].apply(", ".join)

data.head()

In [ ]:
# turn listener into a boolean feature per listener

# A song both of us liked used to appear once per listener, so every song
# level count, mean and correlation below counted it twice. Collapse those
# copies into one row per track and record who liked it in a column each.
listener_names = sorted(data["Listener"].unique())

# True/False per listener for every track
listener_flags = (
    data.assign(Liked=True)
    .pivot_table(
        index="Track URI",
        columns="Listener",
        values="Liked",
        aggfunc="any",
        fill_value=False,
    )
    .rename(columns={name: f"Liked by {name}" for name in listener_names})
    .reset_index()
)

flag_columns = [f"Liked by {name}" for name in listener_names]

# the song level columns are identical across copies, so keep the first one
data = (
    data.drop(columns="Listener")
    .drop_duplicates(subset="Track URI")
    .merge(listener_flags, on="Track URI")
)

# handy summaries of the flags: how many of us liked the song, and who
data["Listener Count"] = data[flag_columns].sum(axis=1)
data["Listeners"] = data[flag_columns].apply(
    lambda row: ", ".join(
        name for name, liked in zip(listener_names, row) if liked
    ),
    axis=1,
)

# Some comparisons below are per listener rather than per song. For those,
# expand back out to one row per track-listener pair; a shared song shows up
# in both listeners' panels, which is what those charts want.
by_listener = pd.concat(
    [
        data[data[f"Liked by {name}"]].assign(Listener=name)
        for name in listener_names
    ],
    ignore_index=True,
)

print(f"{len(data)} unique tracks, {len(by_listener)} track-listener pairs")
print(data["Listeners"].value_counts())
data.head()

In [ ]:
by_listener.groupby("Listener").size()

In [ ]:
data.columns.tolist()

In [ ]:
data.shape

In [ ]:
by_listener.groupby("Listener")["Missing Genres"].sum()

In [ ]:
(
    by_listener.explode("Genres")
    .groupby("Listener")["Genres"]
    .value_counts(dropna=False)
    .groupby(level=0)
    .head(20)
)

In [ ]:
features = [
    "Valence",
    "Energy",
    "Danceability",
    "Acousticness",
    "Tempo",
    "Loudness",
    "Popularity"
]

by_listener.groupby("Listener")[features].describe()

In [ ]:
data.head()

In [ ]:
fig = px.scatter(
    by_listener,
    x="Valence",
    y="Energy",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s) Text", "Genres Text"],
    title="Valence vs Energy of Songs"
)

fig.show()

The graph shows that songs with higher valence often also have higher energy, but there is still a lot of variation. This suggests that valence and energy are somewhat related, but they are measuring different characteristics of a song.

In [ ]:
by_listener.groupby("Listener")[features].corr()

The results show that energy and loudness have the strongest relationship, so songs with more energy also tend to be louder. Energy also decreases as acousticness increases. Valence and energy have a positive but moderate relationship, which supports what we saw in the scatterplot. Popularity does not seem to have a strong relationship with the other audio features.

In [ ]:
fig = px.scatter(
    by_listener,
    x="Energy",
    y="Loudness",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s) Text", "Genres Text"],
    title="Energy vs Loudness of Songs"
)

fig.show()

I made this graph because the correlation results showed that energy and loudness had the strongest relationship in our dataset. The graph makes that relationship easier to see: as songs become more energetic, they generally also become louder. This gives us another useful pair of features to consider when comparing songs.

In [ ]:
# One row per track-genre pair, dropping tracks with no genres
genre_df = by_listener[~by_listener["Missing Genres"]].explode("Genres").copy()
genre_df = genre_df.rename(columns={"Genres": "Genre"})

# Top 5 genres for each listener separately
top_genres = (
    genre_df.groupby("Listener")["Genre"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)
top_genres

In [ ]:
# Keep only each listener's own top 5 genres
top_pairs = set(top_genres.index)

genre_summary = (
    genre_df[
        [
            pair in top_pairs
            for pair in zip(genre_df["Listener"], genre_df["Genre"])
        ]
    ]
    .groupby(["Listener", "Genre"])[["Energy", "Valence"]]
    .mean()
    .reset_index()
)

genre_summary

In [ ]:
fig = px.bar(
    genre_summary,
    x="Genre",
    y=["Energy", "Valence"],
    barmode="group",
    facet_col="Listener",
    title="Average Energy and Valence by Top Genres"
)

fig.update_xaxes(matches=None)
fig.show()

The graph shows that the top genres have different average energy and valence levels. Hip hop has the highest average energy and valence, while R&B has the lowest average energy. Rap, melodic rap, and rage rap are closer together, but still show some differences. This suggests that audio features like energy and valence may help us compare genres and could be useful later when looking for possible subgenres.

### Identifying Similarities and Differences within Genres

In [ ]:
# Lets display the total distribution of genres across all listeners, not just the top 5 for each listener. Using a barplot

# data is already one row per track, so no de-duplication is needed here
unique_tracks = data

# one row per track-genre pair, dropping tracks with no genre label
all_genres = unique_tracks[~unique_tracks["Missing Genres"]].explode("Genres")
all_genres = all_genres.rename(columns={"Genres": "Genre"})

# count how many songs fall under each genre
genre_totals = all_genres["Genre"].value_counts().reset_index()
genre_totals.columns = ["Genre", "Songs"]

# there is a long tail of rare genres, so chart the 20 biggest
top_20 = genre_totals.head(20)

# genre names are long, so put them on the y axis where they have room to
# read left to right instead of being squeezed under the bars
fig = px.bar(
    top_20,
    x="Songs",
    y="Genre",
    orientation="h",
    title="Total Songs per Genre Across All Listeners",
)

# value_counts sorts biggest first, which plots bottom up, so flip the axis
# to put the biggest genre at the top
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=600)

fig.show()

## Finding: Subgenres are often nested within the set, we need to identify common pairings to see if any genres are truly JUST a subgenre.

In [ ]:
# develop a measure of genre co-occurance. Loop through genre pairs and count how many times they appear together out of how many times the appear individually. If one appears together 100% of the time it appears individiually, then they are strongly associated (OR one is a true subgenre). Output should be a table, genre on the left, co-occurance rate top to bottom, and top 3-5 co occuring genres.

# reuse the exploded pairs from above, but a track can list the same genre
# twice when several of its artists share it, so keep one row per pair
genre_pairs = all_genres[["Track URI", "Genre"]].drop_duplicates()

# a genre needs enough songs before its rate means anything
MIN_TRACKS = 20
genre_sizes = genre_pairs["Genre"].value_counts()
common_genres = genre_sizes[genre_sizes >= MIN_TRACKS].index
genre_pairs = genre_pairs[genre_pairs["Genre"].isin(common_genres)]

# the set of songs under each genre, so overlap is just a set intersection
songs_by_genre = genre_pairs.groupby("Genre")["Track URI"].apply(set)

# for each genre, what share of ITS OWN songs also carry each other genre
rows = []
for genre, songs in songs_by_genre.items():
    for other_genre, other_songs in songs_by_genre.items():
        if other_genre == genre:
            continue

        shared = len(songs & other_songs)
        if shared == 0:
            continue

        rows.append({
            "Genre": genre,
            "Co-occurring Genre": other_genre,
            "Genre Songs": len(songs),
            "Shared Songs": shared,
            # 1.00 means every song in Genre also carries the other tag,
            # which is what a true subgenre looks like
            "Co-occurrence Rate": shared / len(songs),
        })

co_occurrence = pd.DataFrame(rows)

# the rate is directional: rap -> melodic rap is not the same as the
# reverse, and that gap is what tells child apart from parent
co_occurrence = co_occurrence.sort_values(
    ["Co-occurrence Rate", "Genre"], ascending=[False, True]
)

# keep each genre's 5 strongest partners
top_co_occurrence = co_occurrence.groupby("Genre").head(5)

top_co_occurrence.round(2)

In [ ]:
# lets maintain a seperate subgenre column for anything we know is a true subgenre (100% co occurance) which is a list of subgenre:parent genre keys.

# a true subgenre: every single one of its songs also carries the other tag
nested = co_occurrence[co_occurrence["Co-occurrence Rate"] == 1.0].copy()

# the parent has to be the broader label, so it needs more songs than the
# child; equal sizes would just mean two names for the same set
nested["Parent Songs"] = nested["Co-occurring Genre"].map(genre_sizes)
nested = nested[nested["Parent Songs"] > nested["Genre Songs"]]

# a subgenre can sit fully inside several parents, so keep the broadest one
nested = nested.sort_values("Parent Songs", ascending=False)
broadest_parent = nested.drop_duplicates(subset="Genre")

# the lookup: subgenre -> parent genre
SUBGENRE_PARENTS = dict(
    zip(broadest_parent["Genre"], broadest_parent["Co-occurring Genre"])
)

# tag every track-genre row, leaving the parent blank for top level labels
genre_pairs["Parent Genre"] = genre_pairs["Genre"].map(SUBGENRE_PARENTS)
genre_pairs["Is Subgenre"] = genre_pairs["Parent Genre"].notna()

# a grouped label that rolls subgenres up but leaves everything else alone,
# so we can compare genres at either level later
genre_pairs["Genre Grouped"] = genre_pairs["Parent Genre"].fillna(
    genre_pairs["Genre"]
)

print(f"{len(SUBGENRE_PARENTS)} true subgenres found")
SUBGENRE_PARENTS

In [ ]:
# For each measurable, I want to see every track color coded by the genre. multicode if a track is in multiple genres. ignore subgenres.

# drop the subgenre rows so a track is not plotted twice under both rap and
# melodic rap; a track still shows up once per genuinely different genre
from IPython.display import HTML, display

top_level = genre_pairs[~genre_pairs["Is Subgenre"]][["Track URI", "Genre"]]

# attach the audio features to each track-genre pair
track_features = unique_tracks[["Track URI", "Track Name"] + features]
genre_tracks = top_level.merge(track_features, on="Track URI")

# the features live in separate columns, but faceting needs them stacked
# into one value column with the feature name alongside
long_features = genre_tracks.melt(
    id_vars=["Track URI", "Track Name", "Genre"],
    value_vars=features,
    var_name="Feature",
    value_name="Value",
)

# squeeze the facets two per row instead of all seven side by side, so each
# panel is wide enough to read
fig = px.violin(
    long_features,
    x="Value",
    y="Genre",
    color="Genre",
    facet_col="Feature",
    facet_col_wrap=2,
    points=False,
    orientation="h",
    hover_data=["Track Name"],
    title="Audio Feature Distributions by Genre",
)

# tempo, loudness and popularity are on completely different scales than
# the 0-1 features, so each panel needs its own x axis
fig.update_xaxes(matches=None, showticklabels=True, title=None)

# the facet labels already name the feature, so drop the "Feature=" prefix
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# the genre names repeat down every panel, so label them once on the left
fig.update_yaxes(title=None)

# every genre needs its own band of vertical space or the violins flatten
# into each other, so size the figure off the genre count instead of
# forcing it into a fixed height
n_genres = long_features["Genre"].nunique()
n_rows = -(-len(features) // 2)
fig.update_layout(
    height=n_rows * max(300, n_genres * 28),
    showlegend=False,
)

# that figure is far taller than the screen, so put it in a fixed-height
# box of its own that scrolls instead of swamping the notebook

display(HTML(
    '<div style="height: 700px; overflow-y: scroll;">'
    + fig.to_html(include_plotlyjs="cdn", full_html=False)
    + "</div>"
))

## Question 1: How are genres similar to and different from each other?

The violin chart above shows every distribution, but with this many genres it is hard to read off an answer. The cells below boil it down to ten parts, each built by a function in `utils/genre_analysis.py` so the Streamlit app can reuse them as is:

1. **Which features separate genres** (share of each feature's variation explained by genre)
2. **Genre sound profiles** (heatmap of average feature values per genre)
3. **Genre map** (every genre placed in 2-D so similar genres sit together)
4. **Most similar / most different genre pairs** plus a head-to-head comparison
5. **How consistent each genre is** (does the label describe one sound or many?)

Parts 6-10 go deeper with machine learning and NLP, and each one is chosen to predict how the clustering (Question 2) and the recommender (Question 3) will behave:

6. **Genre classifier** (random forest): how well sound alone predicts genre, and which genres get confused
7. **Permutation importance**: which features the model actually relies on vs part 1
8. **Genre family tree + silhouette**: how many natural groups exist, and whether genre labels form clusters at all
9. **NLP on track titles**: weighted log-odds vocabulary per genre, and whether titles add information sound misses
10. **Nearest-neighbour purity**: if we recommend the 10 closest songs, how often do they stay in-genre?

All features are standardized (z-scores) first, because tempo and loudness are on much larger scales than the 0-1 features and would otherwise dominate. Popularity is left out since it is not part of how a song sounds.

In [ ]:
from utils.genre_analysis import (
    AUDIO_FEATURES,
    feature_importance,
    genre_profiles,
    genre_distances,
    genre_pairs_ranked,
    nearest_genres,
    genre_spread,
    plot_feature_importance,
    plot_profile_heatmap,
    plot_genre_map,
    plot_genre_compare,
    plot_spread,
)

# one row per track-genre pair (subgenres already rolled out above), with
# every audio feature attached
genre_tracks = (
    top_level.merge(
        unique_tracks[["Track URI", "Track Name", "Artist Name(s) Text"]
                      + AUDIO_FEATURES],
        on="Track URI",
    )
    .dropna(subset=AUDIO_FEATURES)
)

profiles = genre_profiles(genre_tracks, min_tracks=MIN_TRACKS)
distances = genre_distances(profiles)

print(f"{len(genre_tracks)} track-genre rows across {len(profiles)} genres")

### 1. Which audio features differ most between genres?

In [ ]:
importance = feature_importance(genre_tracks)
plot_feature_importance(importance).show()
importance.round(3)

Acousticness, instrumentalness, energy, loudness, speechiness and danceability each have roughly 40-50% of their variation explained by genre, so these are the features that define what a genre "sounds like". Tempo (~9%) and liveness (~15%) barely change between genres, so they are weak signals for telling genres apart (and for clustering or recommending by genre).

### 2. Genre sound profiles

In [ ]:
# a readable spread of familiar genres; swap in any names from profiles.index
SHOWCASE_GENRES = [
    "classical", "ambient", "jazz", "country", "pop", "r-n-b",
    "k-pop", "hip-hop", "rock", "latin", "edm", "metal",
]
SHOWCASE_GENRES = [g for g in SHOWCASE_GENRES if g in profiles.index]

plot_profile_heatmap(profiles, SHOWCASE_GENRES).show()

Each cell is how far the genre's average song sits from the average song overall, in standard deviations (red = higher, blue = lower). Two clear camps show up: a quiet/acoustic camp (classical, ambient, jazz) that is low energy, low loudness and high acousticness, and a loud/produced camp (latin, edm, metal) at the other end. Inside the loud camp, the difference is mostly danceability and valence: hip-hop and latin are danceable and upbeat, metal is energetic but not danceable.

### 3. Genre map

In [ ]:
plot_genre_map(profiles, highlight=SHOWCASE_GENRES).show()

This squeezes all nine features into two axes (PCA). The horizontal axis is essentially loud/energetic vs quiet/acoustic and carries the most information; the vertical axis is mostly danceability and speechiness. Most genres crowd into the loud right half, which is why many popular genres are hard to tell apart on audio features alone, while classical, ambient and similar genres sit far out on their own.

### 4. Most similar and most different genres

In [ ]:
ranked_pairs = genre_pairs_ranked(distances)

print("Most similar genre pairs")
display(ranked_pairs.head(10).round(2))

print("Most different genre pairs")
display(ranked_pairs.tail(10).iloc[::-1].round(2))

In [ ]:
# nearest neighbours for a few genres people actually listen to
for genre in ["hip-hop", "rock", "pop", "edm"]:
    if genre in distances.index:
        print(genre)
        display(nearest_genres(distances, genre, n=5).round(2))

In [ ]:
# head-to-head: which features separate two genres the most
plot_genre_compare(profiles, "metal", "classical").show()
plot_genre_compare(profiles, "hip-hop", "pop").show()

The most similar pairs are mostly ones you would expect to overlap (latino/reggaeton, indie/indie-pop, edm/house, punk/punk-rock), which is a good sanity check that the audio features capture real musical similarity. The most different pairs almost all involve comedy, which is spoken word and so sits far out on speechiness and liveness, or quiet genres like sleep and ambient.

### 5. How consistent is each genre's sound?

In [ ]:
# require more tracks here so a tiny genre does not look consistent by luck
spread = genre_spread(genre_tracks, min_tracks=100)
plot_spread(spread).show()

Genres named after a place or language (german, iranian, french, british) or a use case (sleep, anime, disney, comedy) are the least consistent: the label says where the music comes from or what it is for, not how it sounds. Style-based genres (metalcore, edm, house, pop punk) are the most uniform. This matters for the clustering and recommendation parts: clusters should line up well with style genres but will split up the place/use-case ones.

### 6. Can a model recognise genre from sound alone?

A random forest is trained to predict genre from the nine audio features. Two choices keep the test honest:

- **Only single-genre tracks.** A song tagged both *rock* and *alt-rock* would land in train and test under different labels, leaking the answer.
- **Balanced sample** (up to 300 songs per genre, genres with 100+ songs) so big genres can't inflate accuracy, and a stratified 75/25 split.

With 100+ genres, random guessing is under 1%, so top-5 accuracy is reported alongside top-1.

In [ ]:
from utils.genre_ml import (
    single_genre_sample, split, train_audio_classifier, classifier_report,
    plot_f1, plot_confusion, model_importance, plot_importance_comparison,
    genre_tree, genre_silhouette, distinctive_title_words,
    text_vs_audio_models, text_audio_agreement, plot_model_comparison,
    neighbour_purity, plot_link_chart,
)

sample = single_genre_sample(genre_tracks)
train_idx, test_idx = split(sample)
rf = train_audio_classifier(sample, train_idx)

scores, per_genre_f1, confusion, most_confused = classifier_report(
    rf, sample, test_idx
)
print(f"{len(sample)} songs, {sample['Genre'].nunique()} genres")
scores.round(3)

In [ ]:
plot_f1(per_genre_f1).show()

In [ ]:
# the biggest off-diagonal cells: true genre -> what the model called it
display(most_confused.head(15).round(2))

# a readable block of the full confusion matrix
CONFUSION_GENRES = [
    "black-metal", "death-metal", "grindcore", "heavy-metal", "metalcore",
    "techno", "minimal-techno", "detroit-techno", "chicago-house", "house",
    "classical", "ambient", "new-age", "piano", "sleep",
]
plot_confusion(confusion, CONFUSION_GENRES).show()

Sound alone gets the exact genre right about **25%** of the time and puts it in the top 5 about **50%** of the time, around 30x better than chance with 100+ genres. So audio carries real genre signal, but most genres are not cleanly separable.

The mistakes are not random, they stay inside **families**: metal subgenres get confused with each other, techno with minimal/detroit techno and chicago house, Brazilian styles with each other (samba/pagode, forro/sertanejo), ambient with new-age. Genres named after a language or region (turkish, malay, spanish) and our own small listener genres (rap, r&b) are close to unrecognisable.

**Link to Q2/Q3:** anything the classifier confuses, k-means will put in the same cluster, and the recommender will happily cross between. That is fine (a metalcore fan probably wants death metal suggestions), but cluster-vs-genre agreement scores will look low for these families even when the clusters are musically sensible.

### 7. What does the model actually rely on?

In [ ]:
perm_importance = model_importance(rf, sample, test_idx)
plot_importance_comparison(perm_importance, importance).show()
perm_importance.round(3)

Permutation importance shuffles one feature at a time and measures how much test accuracy falls. Acousticness and instrumentalness stay on top, agreeing with part 1. The interesting change is **tempo**: on its own it barely differs between genres (eta² ~9%), but the model loses ~4 points of accuracy without it. Tempo matters *in combination* (e.g. drum-and-bass vs dubstep are both loud and electronic but sit at different tempos), which a one-feature-at-a-time view can't see. Liveness is the only feature that is weak by both measures.

Energy and loudness rank lower here than in part 1 because they are highly correlated: shuffle one and the model leans on the other.

**Link to Q2/Q3:** k-means and nearest-neighbour search weight every standardized feature equally. Liveness adds mostly noise to those distances, and energy + loudness effectively count the same information twice. Dropping liveness, or down-weighting the energy/loudness pair, is a defensible tweak for both.

### 8. Natural genre families and whether genres form clusters

**Hierarchical clustering** (Ward linkage) on the genre profiles from part 2 builds a family tree from the bottom up: the two most similar genres merge first, and so on. Cutting the tree into 8 branches gives data-driven genre families.

Then the **silhouette score** asks the question from the other side: treating each genre label as if it were a cluster, is a song closer to its own genre than to the nearest other genre? (+1 = perfect, 0 = on the boundary, negative = closer to another genre.)

In [ ]:
tree_fig, families = genre_tree(profiles, n_families=8)
tree_fig.show()

pd.set_option("display.max_colwidth", None)
families

In [ ]:
silhouettes = genre_silhouette(sample)
print(f"mean silhouette across genres: {silhouettes['Silhouette'].mean():.2f}")
display(silhouettes.head(8).round(2))
display(silhouettes.tail(8).round(2))

The tree finds sensible families without being told anything about genre names: a quiet/acoustic branch (classical, ambient, piano, sleep, study), a techno branch, a metal branch, a rap branch (trap, drill, melodic rap, cloud rap), a dance branch that pulls in hip-hop, latin and reggae, and comedy alone.

But the **average silhouette is negative (about -0.15)**. Song for song, most tracks are closer to some *other* genre's centre than their own. Only comedy is clearly positive; minimal-techno, party and study hover around zero; language/region genres (german, iranian, british) and anime are the most negative.

**Link to Q2:** this is the most important result for clustering. Clusters on audio features will **not** reproduce 100+ genre labels, and the right comparison is with roughly 6-10 families like the ones above, not with individual genres. A low adjusted Rand index against raw genre labels is expected, not a sign the clustering failed. Clusters should be described as *refining* genres into sound-based families, which is what the research question asks.

### 9. NLP: what do track titles say about genre?

Titles are the only text we have, so two questions: (a) which words mark each genre, and (b) does adding title text help predict genre beyond the audio?

For (a), plain word counts favour common words and TF-IDF favours one-off words. **Weighted log-odds with an informative Dirichlet prior** (Monroe, Colaresi & Quinn, 2008) compares each genre's word rates to every other genre's while shrinking rare words toward the overall rate, then reports a z-score so a word used twice can't win by luck.

In [ ]:
NLP_GENRES = [
    "classical", "sleep", "comedy", "metal", "country",
    "latin", "hip-hop", "study", "k-pop",
]
NLP_GENRES = [g for g in NLP_GENRES if g in genre_tracks["Genre"].unique()]
distinctive_title_words(genre_tracks, NLP_GENRES)

In [ ]:
# (b) title-only TF-IDF + logistic regression vs audio random forest vs both
model_comparison = text_vs_audio_models(sample, train_idx, test_idx, rf)
plot_model_comparison(model_comparison).show()
display(model_comparison.round(3))

# do genres with similar titles also sound similar?
rho, p_value, text_audio_pairs = text_audio_agreement(genre_tracks, distances)
print(f"Spearman correlation, title similarity vs audio similarity: "
      f"{rho:.2f} (p = {p_value:.1g})")

The vocabulary is genuinely genre-specific: classical titles are full of musical form (*allegro, major, minor, variations*), sleep titles describe noise (*rain, white, brown, waves, loopable*), metal is *dead, dark, devil, hell*, and latin is Spanish function words. One data-quality catch: the *k-pop* label in the public dataset is dominated by Hindi words (*dil, tum, mein, meri*), so many songs under that label are Bollywood, not Korean pop. That label should be treated with caution in every part.

Titles alone predict genre about half as well as audio, but they are **complementary**: averaging the two models' probabilities beats audio alone on both top-1 and top-5. The correlation between title similarity and audio similarity across genre pairs is weak (rho ~0.13): titles mostly encode **language and theme**, while audio encodes **sound**.

**Link to Q3:** this is the case for a **hybrid recommender**. A pure audio recommender will mix languages freely (a Spanish ballad and an English ballad look identical in audio features); adding a small text or language signal would keep recommendations in the listener's language where that matters.

### 10. Will recommendations stay in-genre?

A "more songs like this" recommender is, at its core, a k-nearest-neighbours search. So this runs that search directly: for 30,000 seed songs, find the 10 closest songs in the full standardized audio space and count how many share at least one genre with the seed. **Lift** compares that to picking 10 songs at random.

In [ ]:
overall_purity, purity = neighbour_purity(genre_tracks, k=10)
print(f"on average {overall_purity:.0%} of a song's 10 nearest neighbours "
      f"share a genre with it")
display(purity.head(8).round(3))
display(purity.tail(8).round(3))

Only about **17%** of nearest neighbours share a genre with the seed song, but every genre is well above chance: from about 4x for malay to 75x for comedy. Comedy and sleep keep almost all their neighbours; jazz, reggae and reggaeton keep 40%+; rap, r&b and language genres keep under 5%.

The last chart puts parts 6, 8 and 10 together.

In [ ]:
# one row per genre: classifier F1, neighbour purity and silhouette
link = (
    per_genre_f1
    .merge(purity[["Genre", "Purity"]], on="Genre")
    .merge(silhouettes, on="Genre")
)
# label only the extremes so the chart stays readable
extremes = pd.concat([link.nlargest(8, "Purity"), link.nlargest(4, "F1"),
                      link.nsmallest(3, "Purity")])["Genre"]
link["Label"] = link["Genre"].where(link["Genre"].isin(extremes), "")

r_f1_purity = link[["F1", "Purity"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman correlation, classifier F1 vs neighbour purity: {r_f1_purity:.2f}")
plot_link_chart(link).show()

Genres that are easy to classify are also the ones whose nearest neighbours stay in-genre (Spearman rho ~0.77). Two methods that work very differently, a supervised classifier and an unsupervised neighbour search, agree on which genres have a distinct sound. Colour shows the silhouette from part 8.

**Link to Q2/Q3 in one line:** for genres in the top right (comedy, sleep, jazz, reggaeton, techno), clusters will match the label and recommendations will stay in-genre. For genres in the bottom left (rap, r&b, language and region genres), clusters will split the label across several sound families and recommendations will cross genres. That is the expected behaviour of a sound-based system, not a bug.

### Takeaways for Question 1

- **Differences:** genres split first on loud/energetic vs quiet/acoustic, then on danceability and speechiness. Tempo matters only in combination with other features, and liveness hardly matters at all.
- **Similarities:** genres group into roughly 6-10 sound families (metal, techno, hip-hop/rap, dance/latin, quiet/acoustic...). Inside a family, a classifier confuses genres and k-NN mixes them.
- **Label quality:** genres named by style describe a consistent sound. Genres named by place, language or use case do not, and the *k-pop* label is largely Bollywood in the public data.
- **What this means for Q2:** genre labels are not audio clusters (mean silhouette ~ -0.15). Compare clusters to families, not individual genres.
- **What this means for Q3:** an audio-only recommender stays in-genre for distinct genres and crosses over for the rest. Titles carry complementary information, so a hybrid would improve it.


In [ ]:
# save the track-genre table for the Streamlit app so it does not have to
# rerun the cleaning steps above every time it loads
PROCESSED = DATA_DIR / "processed"
PROCESSED.mkdir(exist_ok=True)
genre_tracks.to_csv(PROCESSED / "genre_tracks.csv.gz", index=False)
print(f"saved {len(genre_tracks)} rows to {PROCESSED / 'genre_tracks.csv.gz'}")